# ARC-AGI-3 VLM Capability Check

ARC-AGI-3 화면을 VLM에게 보여주고, 사람이 하듯이 `목표`, `객체 역할`, `액션 가설`, `다음 행동`을 추론할 수 있는지 확인하는 실험 노트북입니다.

필요한 것: `OPENAI_API_KEY`. 없으면 이미지와 prompt만 확인하고 API 호출은 건너뜁니다.

In [ ]:
from pathlib import Path
import base64
import io
import json
import os
import sys
from pprint import pprint

import numpy as np
from IPython.display import display, Markdown

try:
    from PIL import Image
except Exception as exc:
    Image = None
    print('PIL unavailable:', repr(exc))

IN_KAGGLE = Path('/kaggle/input').exists()
COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
ENV_DIR = next(
    (p for p in [COMP_ROOT / 'environment_files', Path.cwd() / 'environment_files', Path('/kaggle/working/agi-arc/environment_files')] if p.exists()),
    COMP_ROOT / 'environment_files',
)
REPO_ROOT = Path(os.getenv('AGI_ARC_REPO', Path.cwd()))

if IN_KAGGLE and COMP_ROOT.exists():
    wheel_dir = COMP_ROOT / 'arc_agi_3_wheels'
    if wheel_dir.exists():
        !pip install --no-index --find-links {wheel_dir} arc-agi python-dotenv openai -q

for candidate in [REPO_ROOT / 'src', Path.cwd() / 'src', Path('/kaggle/working/agi-arc/src')]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

print('ENV_DIR =', ENV_DIR)
print('REPO_ROOT =', REPO_ROOT)
print('OPENAI_API_KEY set =', bool(os.getenv('OPENAI_API_KEY')))

In [ ]:
ARC_PALETTE = {
    0: (0, 0, 0),
    1: (0, 116, 217),
    2: (255, 65, 54),
    3: (46, 204, 64),
    4: (255, 220, 0),
    5: (170, 80, 220),
    6: (255, 133, 27),
    7: (127, 219, 255),
    8: (135, 135, 135),
    9: (255, 255, 255),
    10: (80, 80, 80),
    11: (180, 180, 180),
    12: (128, 0, 0),
    13: (0, 128, 128),
    14: (128, 128, 0),
    15: (255, 105, 180),
}

def as_list_grid(grid):
    if grid is None:
        return []
    if hasattr(grid, 'tolist'):
        grid = grid.tolist()
    return [[int(cell) for cell in row] for row in grid]

def raw_frame_stack(raw_frame):
    return list(getattr(raw_frame, 'frame', []) or [])

def latest_grid_from_raw(raw_frame):
    frames = raw_frame_stack(raw_frame)
    return as_list_grid(frames[-1]) if frames else []

def summarize_grid(grid):
    grid = as_list_grid(grid)
    if not grid:
        return {'shape': [0, 0], 'colors': [], 'nonzero': 0}
    arr = np.asarray(grid, dtype=int)
    return {
        'shape': list(arr.shape),
        'colors': sorted(int(x) for x in np.unique(arr)),
        'nonzero': int(np.count_nonzero(arr)),
    }

def grid_to_rgb_array(grid):
    grid = as_list_grid(grid)
    if not grid:
        return np.zeros((1, 1, 3), dtype=np.uint8)
    arr = np.asarray(grid, dtype=int)
    rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.uint8)
    for value, color in ARC_PALETTE.items():
        rgb[arr == value] = color
    return rgb

def grid_to_image(grid, scale=14, draw_grid=True):
    rgb = grid_to_rgb_array(grid)
    if Image is None:
        return rgb
    h, w = rgb.shape[:2]
    img = Image.fromarray(rgb, mode='RGB').resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
    if draw_grid and scale >= 8:
        px = img.load()
        line = (35, 35, 35)
        for gx in range(0, w * scale, scale):
            for yy in range(h * scale):
                px[gx, yy] = line
        for gy in range(0, h * scale, scale):
            for xx in range(w * scale):
                px[xx, gy] = line
    return img

def grid_to_png_data_url(grid, scale=14):
    if Image is None:
        raise RuntimeError('PIL Image is required to encode PNG for VLM API calls')
    img = grid_to_image(grid, scale=scale, draw_grid=True)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    encoded = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'data:image/png;base64,{encoded}'

def display_grid(grid, scale=12):
    image = grid_to_image(grid, scale=scale, draw_grid=True)
    if Image is None:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 6))
        plt.imshow(image, interpolation='nearest')
        plt.axis('off')
        plt.show()
    else:
        display(image)

In [ ]:
def collect_game_reset(game_id='bp35', mode='offline', render_mode=None):
    from arc_agi import Arcade, OperationMode

    arcade = Arcade(
        arc_api_key=os.getenv('ARC_API_KEY', 'test-key-123'),
        arc_base_url=os.getenv('ARC_BASE_URL', 'http://gateway:8001' if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 'https://three.arcprize.org'),
        operation_mode=OperationMode(mode.lower()),
        environments_dir=str(ENV_DIR),
        recordings_dir=str(Path('/kaggle/working/recordings') if IN_KAGGLE else Path('recordings')),
    )
    scorecard_id = arcade.create_scorecard(source_url='vlm-capability-check', tags=['vlm-capability-check'])
    wrapper = arcade.make(
        game_id,
        scorecard_id=scorecard_id,
        save_recording=False,
        include_frame_data=True,
        render_mode=render_mode,
    )
    raw = wrapper.reset()
    return arcade, wrapper, scorecard_id, raw

def close_arcade(arcade, scorecard_id):
    try:
        arcade.close_scorecard(scorecard_id)
    except Exception as exc:
        print('close_scorecard failed:', repr(exc))

def frame_metadata(raw_frame, game_id=None):
    grid = latest_grid_from_raw(raw_frame)
    return {
        'game_id': game_id or getattr(raw_frame, 'game_id', None),
        'state': getattr(getattr(raw_frame, 'state', None), 'name', str(getattr(raw_frame, 'state', None))),
        'available_actions': [getattr(a, 'name', str(a)) for a in list(getattr(raw_frame, 'available_actions', []) or [])],
        'levels_completed': getattr(raw_frame, 'levels_completed', None),
        'win_levels': getattr(raw_frame, 'win_levels', None),
        'frame_count': len(raw_frame_stack(raw_frame)),
        'latest_grid': summarize_grid(grid),
    }

# 예시 수집
GAME_ID = os.getenv('ARC_VLM_GAME_ID', 'bp35')
arcade, wrapper, scorecard_id, raw = collect_game_reset(game_id=GAME_ID, mode=os.getenv('ARC_VLM_MODE', 'offline'))
meta = frame_metadata(raw, game_id=GAME_ID)
pprint(meta)
display_grid(latest_grid_from_raw(raw), scale=12)

In [ ]:
SYSTEM_PROMPT = '''You are analyzing an ARC-AGI-3 interactive game from a single grid image.
The grid is a rendered game state. Coordinates use x,y with (0,0) at top-left.
Actions are abstract and game-specific. ACTION6 may require x,y coordinates.
Infer cautiously. Separate visual facts from hypotheses.
Return compact JSON only.'''

def build_initial_prompt(meta):
    return f'''
Analyze this ARC-AGI-3 game screen.

Metadata:
{json.dumps(meta, ensure_ascii=False, indent=2)}

Please return JSON with these keys:
- visual_facts: concrete objects/regions/colors/labels you can see
- likely_game_family: navigation | painting | alignment | transformation | selection | unknown
- likely_goal: best guess of what the player must achieve
- object_roles: list of objects with role guesses such as player, goal, obstacle, reference, tool, button, movable_object
- action_hypotheses: what ACTION3/ACTION4/ACTION6/ACTION7 might do, with confidence
- recommended_probe_actions: first 3 to 5 actions to learn the rules, using strings like ACTION3 or ACTION6|x=10,y=20
- risk_notes: what actions or regions may be UI/reset/danger/no-op
- confidence: 0.0 to 1.0
'''.strip()

def ask_openai_vlm_about_frame(raw_frame, game_id=None, model=None):
    if not os.getenv('OPENAI_API_KEY'):
        print('OPENAI_API_KEY is not set. Showing prompt only; API call skipped.')
        meta = frame_metadata(raw_frame, game_id=game_id)
        print(build_initial_prompt(meta))
        return None

    from openai import OpenAI
    client = OpenAI()
    model = model or os.getenv('OPENAI_VLM_MODEL', 'gpt-4.1')
    meta = frame_metadata(raw_frame, game_id=game_id)
    image_url = grid_to_png_data_url(latest_grid_from_raw(raw_frame), scale=int(os.getenv('ARC_VLM_IMAGE_SCALE', '14')))
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': build_initial_prompt(meta)},
                    {'type': 'image_url', 'image_url': {'url': image_url, 'detail': 'high'}},
                ],
            },
        ],
        temperature=0,
    )
    text = completion.choices[0].message.content
    display(Markdown('```json\n' + text + '\n```'))
    return text

vlm_initial_answer = ask_openai_vlm_about_frame(raw, game_id=GAME_ID)

In [ ]:
def parse_action(action_text):
    from arcengine import GameAction
    if '|' not in action_text:
        return GameAction.from_name(action_text), None
    name, payload_text = action_text.split('|', 1)
    payload = {}
    for part in payload_text.split(','):
        key, value = part.split('=', 1)
        payload[key.strip()] = int(value)
    tool_action = GameAction.from_name(name)
    return tool_action, payload

def step_and_compare(wrapper, action_text):
    before = raw_frame_stack(getattr(step_and_compare, 'last_raw', raw))[-1]
    action, payload = parse_action(action_text)
    after_raw = wrapper.step(action, data=payload)
    step_and_compare.last_raw = after_raw
    after = latest_grid_from_raw(after_raw)
    print('action =', action_text)
    print('after meta:')
    pprint(frame_metadata(after_raw, game_id=GAME_ID))
    print('after image:')
    display_grid(after, scale=12)
    return before, after_raw

def ask_openai_vlm_about_transition(before_grid, action_text, after_raw, game_id=None, model=None):
    if not os.getenv('OPENAI_API_KEY'):
        print('OPENAI_API_KEY is not set. Transition prompt skipped.')
        return None
    from openai import OpenAI
    client = OpenAI()
    model = model or os.getenv('OPENAI_VLM_MODEL', 'gpt-4.1')
    after_grid = latest_grid_from_raw(after_raw)
    meta = frame_metadata(after_raw, game_id=game_id)
    prompt = f'''
You see BEFORE and AFTER screenshots from an ARC-AGI-3 game.
The action taken was: {action_text}
After metadata:
{json.dumps(meta, ensure_ascii=False, indent=2)}

Return compact JSON with:
- observed_change
- likely_action_meaning
- whether_this_helped_goal: true/false/unknown
- updated_goal_hypothesis
- updated_action_hypotheses
- next_best_actions
- confidence
'''.strip()
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': prompt},
                    {'type': 'text', 'text': 'BEFORE image:'},
                    {'type': 'image_url', 'image_url': {'url': grid_to_png_data_url(before_grid), 'detail': 'high'}},
                    {'type': 'text', 'text': 'AFTER image:'},
                    {'type': 'image_url', 'image_url': {'url': grid_to_png_data_url(after_grid), 'detail': 'high'}},
                ],
            },
        ],
        temperature=0,
    )
    text = completion.choices[0].message.content
    display(Markdown('```json\n' + text + '\n```'))
    return text

# 사용 예시:
# before_grid, after_raw = step_and_compare(wrapper, 'ACTION3')
# vlm_transition_answer = ask_openai_vlm_about_transition(before_grid, 'ACTION3', after_raw, game_id=GAME_ID)

In [ ]:
# 실험이 끝나면 scorecard를 닫습니다.
# 여러 action을 더 해보고 싶으면 이 셀은 마지막에 실행하세요.
close_arcade(arcade, scorecard_id)